In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertForTokenClassification
from torch.nn import BCEWithLogitsLoss
from torch.optim import AdamW
from sklearn.metrics import classification_report, precision_recall_fscore_support, accuracy_score

In [ ]:
# configuration for BERT
BATCH_SIZE = 16  
EPOCHS = 10
LEARNING_RATE = 5e-5
MAX_LEN = 128
MODEL_NAME = 'bert-base-uncased'
LABEL_COLUMNS = ['FP', 'RP', 'RV', 'RS', 'PW']

In [ ]:
class DisfluencyDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        # Group by segment to reconstruct full sentences
        # We aggregate the words into a list, and the 5 target columns into a list of lists
        self.data = df.groupby(['fname', 'speaker', 'seg_idx']).agg({
            'word': list,
            'FP': list, 'RP': list, 'RV': list, 'RS': list, 'PW': list
        }).reset_index()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        words = [str(w) for w in row['word']]
        
        # Construct target vectors for each word: shape (num_words, 5)
        # e.g., if word is "um", vector might be [1, 0, 0, 0, 0]
        targets = []
        for i in range(len(words)):
            t = [row[col][i] for col in LABEL_COLUMNS]
            targets.append(t)
            
        # Tokenize
        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            return_offsets_mapping=True,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        # Align labels with tokens
        word_ids = encoding.word_ids(batch_index=0)
        
        # Create a "Labels" tensor of shape (MAX_LEN, 5)
        # and a "Loss Mask" to tell the model which tokens are valid (ignoring padding/subwords)
        encoded_labels = torch.zeros((self.max_len, len(LABEL_COLUMNS)), dtype=torch.float32)
        loss_mask = torch.zeros((self.max_len), dtype=torch.float32)
        
        previous_word_idx = None
        for idx, word_id in enumerate(word_ids):
            # Special tokens (None) or sub-word tokens (same word_id as prev) get ignored
            if word_id is not None and word_id != previous_word_idx:
                # This is the first token of a new word -> Assign the label
                encoded_labels[idx] = torch.tensor(targets[word_id], dtype=torch.float32)
                loss_mask[idx] = 1.0 # This token is valid for calculating loss
            
            previous_word_idx = word_id

        # Flatten tensors for simpler return
        item = {key: val.squeeze() for key, val in encoding.items() if key != 'offset_mapping'}
        item['labels'] = encoded_labels
        item['loss_mask'] = loss_mask
        
        return item

In [ ]:
import random

def get_splits(df):
    """
    Custom split for a non-standard Switchboard subset.
    Splits unique Conversation IDs randomly (85% Train, 15% Test)
    to ensure no speaker overlap between sets.
    """
    # Get unique Conversation IDs (e.g., 'sw2005', 'sw4400')
    # We extract this from 'fname' (e.g., 'sw2005.trans' -> 'sw2005')
    unique_ids = df['fname'].unique()
    n_total = len(unique_ids)
    
    random.seed(42) 
    
    # Convert to list to shuffle
    unique_ids_list = list(unique_ids)
    random.shuffle(unique_ids_list)
    
    # Define split point (e.g., 15% for test)
    split_idx = int(n_total * 0.85)
    
    train_ids = set(unique_ids_list[:split_idx])
    test_ids = set(unique_ids_list[split_idx:])
    
    # Create DataFrames based on these IDs
    train_df = df[df['fname'].isin(train_ids)].copy()
    test_df = df[df['fname'].isin(test_ids)].copy()
    
    print(f"Total Conversations: {n_total}")
    print(f"Train Set: {len(train_ids)} conversations ({len(train_df)} rows)")
    print(f"Test Set:  {len(test_ids)} conversations ({len(test_df)} rows)")
    
    return train_df, test_df

In [ ]:
def evaluate_detailed(model, loader, device, label_columns):
    """
    Runs inference on the test set and prints classification report + Accuracy metrics.
    Returns: Unweighted Average Recall (UAR)
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    print("\nRunning full evaluation on Test Set...")
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            loss_mask = batch['loss_mask'].to(device)

            # Forward pass
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            # Convert logits to binary predictions
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            
            # Masking
            active_mask = loss_mask.view(-1) == 1
            flat_preds = preds.view(-1, len(label_columns))
            flat_labels = labels.view(-1, len(label_columns))
            
            valid_preds = flat_preds[active_mask]
            valid_labels = flat_labels[active_mask]
            
            all_preds.append(valid_preds.cpu().numpy())
            all_labels.append(valid_labels.cpu().numpy())

    # Stack all batches
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    
    #  Classification Report
    print("\n" + "="*60)
    print("FINAL CLASSIFICATION REPORT")
    print("="*60)
    print(classification_report(all_labels, all_preds, target_names=label_columns, zero_division=0))
    
    # UAR
    _, recall, _, _ = precision_recall_fscore_support(all_labels, all_preds, average=None, zero_division=0)
    uar = np.mean(recall)
    
    # Accuracy Metrics
    # Exact Match: Did we get the entire vector [0, 1, 0, 1, 0] correct?
    subset_acc = accuracy_score(all_labels, all_preds)
    
    # Hamming Accuracy: What % of individual cells are correct? (Includes correctly predicting '0')
    # This will be very high (90%+) because most labels are 0.
    hamming_acc = (all_preds == all_labels).mean()

    print("-" * 60)
    print(f"Unweighted Average Recall (UAR):   {uar:.4f}")
    print(f"Subset Accuracy (Exact Match):     {subset_acc:.4f}")
    print(f"Hamming Accuracy (Element-wise):   {hamming_acc:.4f}")
    print("-" * 60)
    
    return uar
            
   

def evaluate_loss(model, loader, criterion, device):
    """Simple evaluation loop just for calculating Test Loss during training."""
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            loss_mask = batch['loss_mask'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss_unreduced = criterion(outputs.logits, labels)
            
            loss_per_token = loss_unreduced.mean(dim=2)
            masked_loss = loss_per_token * loss_mask
            final_loss = masked_loss.sum() / (loss_mask.sum() + 1e-9)
            total_loss += final_loss.item()
            
    return total_loss / len(loader)

In [ ]:
# 1. Load Data
print("Loading data...")
df = pd.read_csv('/kaggle/input/switchboard-preprocessed-dataset/PreProcessedData/transcripts.csv')

# Apply Splits
train_df, test_df = get_splits(df)

# Setup Tokenizer and Datasets
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

train_dataset = DisfluencyDataset(train_df, tokenizer, MAX_LEN)
test_dataset = DisfluencyDataset(test_df, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Setup Model
model = BertForTokenClassification.from_pretrained(MODEL_NAME, num_labels=len(LABEL_COLUMNS))
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = BCEWithLogitsLoss(reduction='none')

print(f"Starting training on {device} for {EPOCHS} epochs...")
best_test_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    
    for batch in train_loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        loss_mask = batch['loss_mask'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        loss_unreduced = criterion(outputs.logits, labels)
        
        # Loss calculation logic
        loss_per_token = loss_unreduced.mean(dim=2)
        masked_loss = loss_per_token * loss_mask
        final_loss = masked_loss.sum() / (loss_mask.sum() + 1e-9)
        
        final_loss.backward()
        optimizer.step()
        
        train_loss += final_loss.item()

    avg_train_loss = train_loss / len(train_loader)
    
    # --- EVALUATION LOOP (LOSS ONLY) ---
    avg_test_loss = evaluate_loss(model, test_loader, criterion, device)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f}")

    if avg_test_loss < best_test_loss:
        best_test_loss = avg_test_loss
        torch.save(model.state_dict(), '/kaggle/working/BERT1gb_best.pt')
        print("  -> New best model saved!")

In [ ]:
print("\nTraining Complete. Generating Final Report...")
evaluate_detailed(model, test_loader, device, LABEL_COLUMNS)

In [ ]:
torch.save(model.state_dict(), '/kaggle/working/BERT.pt')
print("Model saved to /kaggle/working/BERT1gb_final.pt")